In [110]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots

import torch
from botorch.models import SingleTaskGP
from botorch.fit import fit_gpytorch_mll
from botorch.optim import optimize_acqf
from botorch.sampling import SobolQMCNormalSampler
from gpytorch.mlls import ExactMarginalLogLikelihood
from botorch.models.transforms import Normalize, Standardize
from botorch.utils.sampling import draw_sobol_samples

In [111]:
# from run_ML_optimization_activity_stability_Ni_Mo_Andreas import 


#############################################################################################
# Helper functions 
#############################################################################################

# names in the save files -> names in the code
# names in json: 
# "Current density mA/cm2": 22.0,
# "Dep time [s]": 249.0,
# "Dep electrolye T [C]": 33.8,
# "Conc Mo/Ni 10:1 liquid": 0.078,
# "Conc Ni/Mo 10:1 liquid": 0.162,
# "Conc H2SO4": 0.05
# "Integrated stability at 10 [mA/cm2]"
variable_names = {
    "Integrated stability at 10 [mA/cm2]": "stability_slope",
    "Deposition current density [mA/cm2]": "current_density",
    "Current density mA/cm2": "current_density",
    "Dep time [s]": "deposition_time",
    "Dep electrolye T [C]": "temperature",
    "Deposition composition mol / L": "concentrations",
    # "pH_regulation": "pH_regulation",
    "NiSO4": "NiSO4",
    "Na2MoO4": "Na2MoO4",
    "Conc H2SO4": "H2SO4",
    "Conc Ni/Mo 10:1 liquid": "liquid1",
    "Conc Mo/Ni 10:1 liquid": "liquid2",
    # 'integrated_area', 'NiSO4 (mol/L)', 'Na2Mo (mol/L)', 'H2SO4 (mol/L)', 'Dep t (s)', 'Dep I (mA/cm²)', 'Dep T (C)'
    "integrated_area": "stability_slope",
    "NiSO4 (mol/L)": "liquid1",
    "Na2Mo (mol/L)": "liquid2",
    "H2SO4 (mol/L)": "H2SO4",
    "Dep t (s)": "deposition_time",
    "Dep I (mA/cm²)": "current_density",
    "Dep T (C)": "temperature",
}

human_names = {
    "current_density": "Current density [mA/cm2]",
    "deposition_time": "Deposition time [s]",
    "temperature": "Temperature [C]",
    "liquid1": "NiSO4 [mol/L]",
    "liquid2": "Na2MoO4 [mol/L]",
    "H2SO4": "H2SO4 [mol/L]",
    "stability_slope": "Integrated stability at 10 [mA/cm2]",
}

# tensors of X, Y, bounds, buckets, etc. need to be stacked in this order
variable_order = [
    "current_density",
    "deposition_time",
    "temperature",
    "liquid1", 
    "liquid2",
    "H2SO4",
    # "NiSO4", "Na2MoO4",
]

# min and max values for the variables
parameter_bounds = {
    "current_density": [1, 200],  # in mA/cm2
    "deposition_time": [60, 600],  # in seconds
    "pH_regulation": [0, 1.5],  # in ml
    "temperature": [30, 70],  # in degrees C
    "liquid1": [0.002, 0.4],  # in mol/L
    "liquid2": [0.002, 0.4],  # in mol/L
    "NiSO4": [0.002, 0.4],  # in mol/L
    "Na2MoO4": [0.002, 0.4],  # in mol/L
    "H2SO4": [0, 0.1],  # in mol/L
}


# how precise the variables can be changed
parameter_granularity = {
    "current_density": -1,  # in mA/cm2
    "deposition_time": -1,  # in seconds
    "pH_regulation": 0.075,  # in ml
    "temperature": 1,  # in degrees C
    "NiSO4": 0.002,  # 1/200 * 0.4 in mol/L
    "Na2MoO4": 0.002,  # 1/200 * 0.4 in mol/L
    "H2SO4": 0.005,  # 1/200 * 1 in mol/L
}

# buckets are the possible values for the variables
buckets = {
    "current_density": None,  # in mA/cm2
    "deposition_time": None,  # in seconds
    # "pH_regulation": torch.arange(0, 1.5, 0.075),  # in ml
    "temperature": torch.arange(30, 70, 1),  # in degrees C
    # "NiSO4": torch.arange(0, 0.4, 0.002),  # 1/200 * 0.8 in mol/L
    # "Na2MoO4": torch.arange(0, 0.4, 0.002),  # 1/200 * 0.8 in mol/L
    "liquid1": torch.arange(0.002, 0.4, 0.002),  # 1/200 * 0.4 in mol/L
    "liquid2": torch.arange(0.002, 0.4, 0.002),  # 1/200 * 0.4 in mol/L
    "H2SO4": torch.arange(0, 0.1, 0.05),  # 1/20 * 0.1 in mol/L
}

# the three input compounds are solved in water at a certain concentration
compound_concentrations = {
    "NiSO4": 0.4,  # in mol/L
    "Na2MoO4": 0.4,  # in mol/L
    "H2SO4": 1,  # in mol/L
}
# list is easier to handle
stock_concentrations = [0.4, 0.4, 1]

# Define the bounds for the variables
bounds = torch.tensor([parameter_bounds[v] for v in variable_order], dtype=torch.double)

# Constraint is like this
# Conc_stock_liquid_1 = 0.4 
# Conc_stock_liquid_2 = 0.4 
# Conc_stock_H2SO4 = 1 
# Sum of Conc_liquid_1 / Conc_stock_liquid_1 + Conc_liquid_2 / Conc_stock_liquid_2 + Conc_H2SO4 / Conc_stock_H2SO4 < 1
# Sum of Conc_liquid_1 / 0.4 + Conc_liquid_2 / 0.4 + Conc_H2SO4 / 1 < 1
# Conc_liquid_1 * 2.5 + Conc_liquid_2 * 2.5 + Conc_H2SO4 * 1 < 1
# list of tuples (indices, coefficients, rhs)
# \sum_i (X[indices[i]] * coefficients[i]) >= rhs
# switch between <= and >= constraints by flipping the sign of the coefficients
inequality_constraints = [
    (
        # indices of the variables we want to constrain
        torch.tensor(
            [variable_order.index("liquid1"), variable_order.index("liquid2"), variable_order.index("H2SO4")],
            dtype=torch.long,
        ),
        # coefficients of the linear combination (weighted sum)
        # Conc_liquid_1 * 2.5 + Conc_liquid_2 * 2.5 + Conc_H2SO4 * 1 <= 1
        -1 * torch.tensor([1.0/0.4, 1.0/0.4, 1.0], dtype=torch.double),
        # bigger or equal to
        -1.0,
    )
]


In [112]:
# load data
data = pd.read_csv("data.csv", delimiter=";")

# rename the columns
data = data.rename(columns=variable_names)

ycol = "stability_slope"
xcols = [_c for _c in data.columns if _c != ycol]

# reorder the columns based on variable_order
data = data[variable_order + [ycol, "experiment"]]

print(variable_order)
# data.iloc[:3][xcols]
# get_iloc_ordered(data, 3)
# data

['current_density', 'deposition_time', 'temperature', 'liquid1', 'liquid2', 'H2SO4']


,current_density,deposition_time,temperature,liquid1,liquid2,H2SO4,stability_slope,experiment
0,36.0,156.0,33.5,0.124,0.274,0.040,0.964289,1
1,162.0,597.0,56.4,0.086,0.345,0.015,0.767740,2
2,62.0,388.0,59.2,0.290,0.051,0.030,0.825400,3
3,182.0,94.0,55.1,0.236,0.158,0.000,0.931936,4
4,8.0,534.0,64.4,0.146,0.230,0.055,0.918212,5
5,165.0,296.0,51.0,0.124,0.187,0.015,0.913022,6
6,177.0,438.0,47.7,0.050,0.159,0.085,0.904954,7
7,77.0,251.0,58.5,0.080,0.244,0.015,0.884511,8
8,159.0,212.0,62.1,0.199,0.065,0.000,0.741278,9
9,70.0,591.0,63.8,0.018,0.045,0.080,0.295755,10


In [114]:
def get_torch_from_df(_df, column_names):
    cols_values = []
    for v in column_names:
        cols_values.append(_df[v].values)
    _data = torch.tensor(np.array(cols_values).T, dtype=torch.double)
    assert _data.shape[1] == len(column_names), f"Expected {len(column_names)} columns, got {_data.shape[1]}"
    assert _data.shape[0] == _df.shape[0], f"Expected {_df.shape[0]} rows, got {_data.shape[0]}"
    return _data

get_torch_from_df(data, variable_order).shape


torch.Size([35, 6])

# Is the BayesOpt model learning?
## Note: not yet, since this is exploration data, where the model intentionally has high uncertainty
Plots to illustrate the learning process

In [115]:
# we didn't save the model during the optimization process
# so we need to load the data and re-run the optimization at each iteration
# only redo this if you have new data

data_with_predictions_filename = "data_with_predictions.csv"
load_from_file = True # only set to False if you have new data, set back to true after running the optimization once

if load_from_file:
    data = pd.read_parquet(data_with_predictions_filename)
    print(f"Loaded from {data_with_predictions_filename}")
    
else:
    def get_my_gp(_x, _y):
        # Define the GP model
        return SingleTaskGP(
            train_X=_x,
            train_Y=_y,
            outcome_transform=Standardize(m=1),
            input_transform=Normalize(d=6, bounds=bounds.T),
        )

    # add new columns to the data
    data["predicted_mean"] = None
    data["predicted_std"] = None

    xdatatensor = get_torch_from_df(data, variable_order)
    ydatatensor = torch.tensor(data[ycol].values, dtype=torch.double).unsqueeze(-1)

    for i in range(1, len(data)):
        
        # get data up to i-1
        # as numpy of shape [n_experiments, n_variables]
        xdata = xdatatensor[:i]
        ydata = ydatatensor[:i]
        
        # Refit the GP model with the new experimental data
        gp = get_my_gp(xdata, ydata)
        mll = ExactMarginalLogLikelihood(gp.likelihood, gp)
        mll = fit_gpytorch_mll(mll)
        
        # get the i-th experiment
        suggested_experiment = xdatatensor[i]
        print(i, suggested_experiment)
        
        posterior = gp.posterior(suggested_experiment.unsqueeze(0))
        predicted_mean = posterior.mean  # Predicted y-value (mean)
        predicted_std = posterior.variance.sqrt()  # Prediction uncertainty (std)
        
        # update the data
        data.loc[i, "predicted_mean"] = predicted_mean
        data.loc[i, "predicted_std"] = predicted_std

    # compute the prediction error
    data["prediction_error"] = data["predicted_mean"] - data[ycol]

    # Convert tensor data to numpy/float before plotting
    data = data.copy()
    data['predicted_mean'] = data['predicted_mean'].apply(lambda x: float(x) if x is not None else None)
    data['predicted_std'] = data['predicted_std'].apply(lambda x: float(x) if x is not None else None)
    data['prediction_error'] = data['prediction_error'].apply(lambda x: float(x) if x is not None else None)
    
    data.to_parquet(data_with_predictions_filename)
    print(f"Saved to {data_with_predictions_filename}")



Loaded from data_with_predictions.csv


In [116]:
# Plot prediction error
fig = px.scatter(
    data.reset_index(), 
    x=data.index,
    y='prediction_error',
    color='stability_slope',  # color by actual outcome
    title='Prediction Error vs Experiment Number'
)
fig.update_layout(
    xaxis_title="Experiment Number",
    yaxis_title="Prediction Error",
    showlegend=True,
    margin=dict(l=0, r=0, t=30, b=0)  # Remove whitespace around plot
)
fig.show()

In [117]:

# Plot relative prediction error
data["prediction_error_relative"] = data["prediction_error"] / data["stability_slope"]
fig = px.scatter(
    data.reset_index(), 
    x=data.index,
    y='prediction_error_relative',
    color='stability_slope',  # color by actual outcome
    title='Prediction Error vs Experiment Number'
)
fig.update_layout(
    xaxis_title="Experiment Number",
    yaxis_title="Prediction Error",
    showlegend=True,
    margin=dict(l=0, r=0, t=30, b=0)  # Remove whitespace around plot
)
fig.show()

In [118]:
# plot the uncertainty over the experiments
xmin = 15
fig = px.scatter(
    data.reset_index().iloc[xmin:],
    x=data.index[xmin:], # 'experiment',
    y='predicted_std',
    color='stability_slope',  # color by actual outcome
    title='Model Uncertainty vs Experiment Number'
)
fig.update_layout(
    xaxis_title="Experiment Number",
    yaxis_title="Predicted Standard Deviation",
    showlegend=True,
    margin=dict(l=0, r=0, t=30, b=0),  # Remove whitespace around plot
)
fig.show()


# What are the best parameters?
Plots to illustrate region of interest / relationships between variables
Drawn from the posterior distribution (final belief of the model)

In [119]:
# final state of the model
xdatatensor = get_torch_from_df(data, variable_order)
ydatatensor = torch.tensor(data[ycol].values, dtype=torch.double).unsqueeze(-1)
best_gp = get_my_gp(xdatatensor, ydatatensor)

In [120]:
# pick parameters at best y value, lowest is best
best_data = data.sort_values(by="stability_slope", ascending=True)
best_xdata_tensor = get_torch_from_df(best_data, variable_order)
best_ydatatensor = torch.tensor(best_data[ycol].values, dtype=torch.double).unsqueeze(-1)

In [121]:
# plot predicted mean and uncertainty interval of temp while fixing the other variables at the best values
for var in variable_order:
    var_idx = variable_order.index(var)
    temps = torch.linspace(bounds[var_idx, 0], bounds[var_idx, 1], 100)
    best_inputs = best_data.iloc[0]
    means = []
    uncertainties = []
    for _t in temps:
        _input = best_xdata_tensor[0].clone()
        _input[var_idx] = _t
        posterior = best_gp.posterior(_input.unsqueeze(0))
        means.append(posterior.mean.item())
        uncertainties.append(posterior.variance.sqrt().item())
    fig = px.line(
        x=temps,
        y=means,
        error_y=uncertainties,
    )
    fig.update_layout(
        title=f"Predicted {human_names[ycol]} vs {human_names[var]}",
        xaxis_title=human_names[var],
        yaxis_title=f"Predicted {human_names[ycol]}",
        showlegend=True,
        margin=dict(l=0, r=0, t=30, b=0),  # Remove whitespace around plot
    )
    fig.show()


In [ ]:
# # plot predicted mean and uncertainty interval of temp while fixing the other variables at the best values
# # same plot as above, but with uncertainty as shaded region
# for var in variable_order:
#     var_idx = variable_order.index(var)
#     temps = torch.linspace(bounds[var_idx, 0], bounds[var_idx, 1], 100)
#     best_inputs = best_data.iloc[0]
#     means = []
#     uncertainties = []
#     for _t in temps:
#         _input = best_xdata_tensor[0].clone()
#         _input[var_idx] = _t
#         posterior = best_gp.posterior(_input.unsqueeze(0))
#         means.append(posterior.mean.item())
#         uncertainties.append(posterior.variance.sqrt().item())

#     fig = go.Figure()
#     fig.add_trace(go.Scatter(
#         x=temps,
#         y=means,
#         mode='lines',
#         name='Mean prediction'
#     ))
#     fig.add_trace(go.Scatter(
#         x=temps,
#         y=[m + c for m, c in zip(means, uncertainties)],
#         mode='lines',
#         line=dict(width=0),
#         showlegend=False
#     ))
#     fig.add_trace(go.Scatter(
#         x=temps,
#         y=[m - c for m, c in zip(means, uncertainties)],
#         mode='lines',
#         line=dict(width=0),
#         fillcolor='rgba(68, 134, 255, 0.3)',
#         fill='tonexty',
#         name='±1 std'
#     ))
#     fig.update_layout(
#         title=f"Predicted {human_names[ycol]} vs {human_names[var]}",
#         xaxis_title=human_names[var],
#         yaxis_title=f"Predicted {human_names[ycol]}",
#         showlegend=True,
#         margin=dict(l=0, r=0, t=30, b=0),  # Remove whitespace around plot
#     )
#     fig.show()

In [124]:
# plot a heatmap of T vs Mo, while fixing the other variables at the best values
# color by the predicted stability slope
params_to_plot = [
    ["temperature", "liquid2"],
    ["temperature", "liquid1"],
    ["current_density", "liquid2"],
]
for params in params_to_plot:
    npoints = 50
    yvals = torch.linspace(bounds[variable_order.index(params[0]), 0], bounds[variable_order.index(params[0]), 1], npoints)
    xvals = torch.linspace(bounds[variable_order.index(params[1]), 0], bounds[variable_order.index(params[1]), 1], npoints)
    best_inputs = best_data.iloc[0]
    means = np.zeros((len(yvals), len(xvals)))
    uncertainties = np.zeros((len(yvals), len(xvals)))
    for i, _y in enumerate(yvals):
        for j, _x in enumerate(xvals):
            _input = best_xdata_tensor[0].clone()
            _input[variable_order.index(params[0])] = _y
            _input[variable_order.index(params[1])] = _x
            posterior = best_gp.posterior(_input.unsqueeze(0))
            means[i, j] = posterior.mean.item()
            uncertainties[i, j] = posterior.variance.sqrt().item()
            
    fig = go.Figure()
    fig.add_trace(
        go.Heatmap(z=means, colorscale='Viridis'),
    )
    fig.update_layout(
        title=f'Predicted Stability Slope vs {params[0]} and {params[1]}',
        yaxis_title=human_names[params[0]],
        xaxis_title=human_names[params[1]],
        margin=dict(l=0, r=0, t=30, b=0),  # Remove whitespace around plot
    )
    
    fig.show()

In [127]:
# plot a heatmap of T vs Mo, while fixing the other variables at the best values
# color by the predicted stability slope
params_to_plot = [
    ["temperature", "liquid2"],
    ["temperature", "liquid1"],
    ["current_density", "liquid2"],
]
for params in params_to_plot:
    npoints = 50
    yvals = torch.linspace(bounds[variable_order.index(params[0]), 0], bounds[variable_order.index(params[0]), 1], npoints)
    xvals = torch.linspace(bounds[variable_order.index(params[1]), 0], bounds[variable_order.index(params[1]), 1], npoints)
    best_inputs = best_data.iloc[0]
    means = np.zeros((len(yvals), len(xvals)))
    uncertainties = np.zeros((len(yvals), len(xvals)))
    for i, _y in enumerate(yvals):
        for j, _x in enumerate(xvals):
            _input = best_xdata_tensor[0].clone()
            _input[variable_order.index(params[0])] = _y
            _input[variable_order.index(params[1])] = _x
            posterior = best_gp.posterior(_input.unsqueeze(0))
            means[i, j] = posterior.mean.item()
            uncertainties[i, j] = posterior.variance.sqrt().item()
            
    # Create subplot with 2 side-by-side heatmaps
    fig = make_subplots(rows=1, cols=2, subplot_titles=('Predicted Mean', 'Uncertainty (±1σ)'))
    
    # Add mean heatmap
    fig.add_trace(
        go.Heatmap(z=means, colorscale='Viridis', showscale=True),
        row=1, col=1
    )
    
    # Add uncertainty heatmap 
    fig.add_trace(
        go.Heatmap(z=uncertainties, colorscale='Viridis', showscale=True),
        row=1, col=2
    )

    fig.update_layout(
        title=f'Predicted Stability Slope vs {params[0]} and {params[1]}',
        margin=dict(l=0, r=0, t=40, b=0),  # Remove whitespace around plot
    )
    
    fig.update_xaxes(title_text=human_names[params[1]], row=1, col=1)
    fig.update_xaxes(title_text=human_names[params[1]], row=1, col=2)
    fig.update_yaxes(title_text=human_names[params[0]], row=1, col=1)
    fig.update_yaxes(title_text=human_names[params[0]], row=1, col=2)
    
    fig.show()